In [1]:

import json
import pennylane as qml
import pennylane.numpy as pnp
import numpy as np

class AbsMagnetization(qml.measurements.StateMeasurement):
    def process_state(self, state, wire_order):
        probs = qml.math.abs(state) ** 2
        expectation = 0
        num_wires = len(wire_order)
        for i in range(len(state)):
            expval = 0
            for j in range(num_wires):
                expval += (-1) ** ((i >> j) & 1)
            expectation += np.abs(expval) * probs[i]
        return expectation / num_wires

def tfim_ground_state(num_qubits, h):
    wires = list(range(num_qubits))
    obs = [qml.PauliZ(i) @ qml.PauliZ(i+1) for i in range(num_qubits-1)]
    obs.extend([qml.PauliX(i) for i in range(num_qubits)])
    coeffs = [-1 for _ in range(len(obs)-num_qubits)]
    coeffs.extend([-h for _ in range(num_qubits)])
    H = qml.Hamiltonian(coeffs, obs)
    H_mat = qml.matrix(H)(wires=wires)
    eigvals, eigvecs = np.linalg.eigh(H_mat)
    return pnp.array(eigvecs[:, 0])

dev = qml.device("default.qubit")

@qml.qnode(dev)
def magnetization(num_qubits, h):
    qml.StatePrep(tfim_ground_state(num_qubits, h), wires=list(range(num_qubits)))
    return AbsMagnetization(wires=list(range(num_qubits)))

def critical_point_estimate(mags, h_values):
    differences = [np.abs(mags[i] - mags[i + 1]) for i in range(len(mags) - 1)]
    ind = np.argmax(np.array(differences))
    h_c = np.mean([h_values[ind], h_values[ind + 1]])
    return h_c

def run(test_case_input: str) -> str:
    num_qubits = json.loads(test_case_input)
    h_values = pnp.arange(0.2, 1.1, 0.005)
    mags = []
    for h in h_values:
        mags.append(magnetization(num_qubits, h))
    output = critical_point_estimate(np.array(mags), h_values)
    return str(output)

def check(solution_output: str, expected_output: str) -> None:
    solution_output = json.loads(solution_output)
    expected_output = json.loads(expected_output)
    assert np.isclose(solution_output, expected_output, rtol=5e-3)

test_cases = [
    ('5', '0.6735'),
    ('2', '0.3535')
]

for i, (input_, expected_output) in enumerate(test_cases):
    print(f"Running test case {i} with input '{input_}'...")
    try:
        output = run(input_)
    except Exception as exc:
        print(f"Runtime Error. {exc}")
    else:
        if not np.isclose(json.loads(output), json.loads(expected_output), rtol=5e-3):
            print(f"Wrong Answer. Have: '{output}'. Want: '{expected_output}'.")
        else:
            print("Correct!")


Running test case 0 with input '5'...
Runtime Error. module 'pennylane' has no attribute 'QubitStateVector'
Running test case 1 with input '2'...
Runtime Error. module 'pennylane' has no attribute 'QubitStateVector'
